## 1. 머신러닝 모델 하이퍼 파라미터 튜닝

In [ ]:
'''
하이퍼 파라미터란 모델 학습시에 사용자가 사전에 설정하는 값을 의미한다.
하이퍼 파라미터의 설정에 따라 모델의 성능이 좌우된다.

하이퍼 파라미터를 잘못 설정한 경우
    Underfitting
    모델이 학습을 충분히 하지 못한 상황
    Overfitting
    모델이 학습 데이터에 과하게 fit되어 일반화가 안되는 상황
    

'''

### 1-1. 회귀 모델 하이퍼 파라미터

In [ ]:
# 선형회귀 하이퍼 파라미터

'''
L1 정규화
모델의 가중치가 과도하게 커지는 것을 막아 과적합 예방

장점
변수를 선택, 제거할 수 있어 희소한 모델을 생성할 수 있고 선형 모델과 잘 
Lasso Regression



L2 정규화
모델의 가중치가 너무 커지지 않게 막아 과적합 예방

장점 
모든 변수의 계수를 조금씩 줄이는 방식으로 모든 변수를 사용함
변수를 제거하지 않아 모델이 비교적 안정적임 - 해석보다는 예측 성능 중심


ElasticNet
L1과 L2정규화를 결합한 형태
필요없는 변수는 제거하고(L1) 나머지 변수들은 균형있게 조정(L2)
시간이 부족한 상황이나 시작 단계에서 사용하고
이후 상황에 따라 L1, L2, ElasticNet 가운데 선택하면 된다.

    L1      - 변수 중 일부만이 의미 있는 경우 (희소한 해석이 필요)
    L2      - 모든 변수가 조금씩 중요할 때
    Elastic - 변수의 수가 많고 변수들 간의 상관관계가 높은 경우
'''

##### 선형회귀

In [ ]:
import pandas as pd
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# 데이터 로드 및 전처리
df = pd.read_csv('datasets/Clean_Dataset.csv')
df = df.drop(['flight', 'departure_time', 'stops', 'arrival_time'], axis=1) # 학습에 필요 없는 문자열 열 제거
df = pd.get_dummies(df, columns=['airline', 'source_city', 'destination_city', 'class'], drop_first=True) # 원 핫 인코딩
X = df.drop('price', axis=1)  # 독립 변수
y = df['price']  # 종속 변수

# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Ridge Regression (L2 정규화)
ridge = Ridge(alpha=1.0)  # L2 정규화 강도
ridge.fit(X_train, y_train)
ridge_preds = ridge.predict(X_test)
ridge_mse = mean_squared_error(y_test, ridge_preds)

# Lasso Regression (L1 정규화)
lasso = Lasso(alpha=0.1)  # L1 정규화 강도
lasso.fit(X_train, y_train)
lasso_preds = lasso.predict(X_test)
lasso_mse = mean_squared_error(y_test, lasso_preds)

print("Ridge Regression MSE:", ridge_mse)
print("Lasso Regression MSE:", lasso_mse)


Ridge Regression MSE: 50508878.307652056
Lasso Regression MSE: 50508855.78114253


##### 랜덤포레스트 - 동시학습 후 투표

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# 데이터 로드 및 전처리
df = pd.read_csv('datasets/Clean_Dataset.csv')
df = df.drop(['flight', 'departure_time', 'stops', 'arrival_time'], axis=1) # 학습에 필요 없는 문자열 열 제거
df = pd.get_dummies(df, columns=['airline', 'source_city', 'destination_city', 'class'], drop_first=True)
X = df.drop('price', axis=1)
y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 랜덤 포레스트 모델
rf = RandomForestRegressor(
    n_estimators=100,      # 트리 개수 (투표할 결정 트리의 수 - 많을 수록 안정적)
    max_depth=10,          # 각 트리의 최대 깊이 (깊을수록 복잡한 학습 / 과적합 위험)
    min_samples_split=5,   # 노드를 분할하기 위한 최소 샘플 수 (작을 수록 복잡한 모델)
    min_samples_leaf=2,    # 리프 노드에 있어야 하는 최소 샘플 수 (의결정족수)
    random_state=42        # 결과 재현성을 위한 설정
)
rf.fit(X_train, y_train)  # 모델 학습
rf_preds = rf.predict(X_test)  # 테스트 데이터 예측
rf_mse = mean_squared_error(y_test, rf_preds)  # MSE 계산

print("Random Forest MSE:", rf_mse)


Random Forest MSE: 20024154.727510292


##### 그래디언트 부스트 - 순차적인 학습

In [7]:
from sklearn.ensemble import GradientBoostingRegressor

# 데이터 로드 및 전처리
df = pd.read_csv('datasets/Clean_Dataset.csv')
df = df.drop(['flight', 'departure_time', 'stops', 'arrival_time'], axis=1) # 학습에 필요 없는 문자열 열 제거
df = pd.get_dummies(df, columns=['airline', 'source_city', 'destination_city', 'class'], drop_first=True)
X = df.drop('price', axis=1)
y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 그래디언트 부스트 모델
gb = GradientBoostingRegressor(
    learning_rate=0.1,     # 학습률 (랜덤포레스트 + 지속적인 학습 = 그래디언트 부스트) : 작을수록 신중한 학습 (일반화 좋음)
    n_estimators=100,      # 트리 개수
    max_depth=10,           # 각 트리의 최대 깊이
    random_state=42        # 결과 재현성을 위한 설정
)
gb.fit(X_train, y_train)  # 모델 학습
gb_preds = gb.predict(X_test)  # 테스트 데이터 예측
gb_mse = mean_squared_error(y_test, gb_preds)  # MSE 계산

print("Gradient Boosting MSE:", gb_mse)


KeyboardInterrupt: 

### 1-2. 분류 모델 하이퍼 파라미터

##### 의사결정나무

In [ ]:
'''
의사결정나무는 데이터를 여러 질문을 통해 분기하며 맨 위 잎에 클레스를 두고 예측하는 모델


주요 하이퍼 파라미터

    max_depth
        트리의 최대 깊이(최대 질문 수)
        깊을수록 복잡한 규칙을 학습하지만 과적합 위험성이 있음
        
    min_samples_split
        노드를 분할하기 위해 필요한 최소 샘플 수 (분할 시도 전에 파악하는 sample 수)
        이 수 이하의 샘플을 가지면 분화하지 않음
        
    min_samples_leaf
        리프노드에 있어야 하는 최소 샘플 수 (분할 수에 자식 노드의 sample 수 파악)
        자식 노드가 이 수 이하로 있으면 분할을 취소함
        
    max_features
        분기할 때 고려할 특성의 개수
        질문을 던질 특성의 개수
        'auto', 'sqrt', 5
    
여기서 min_sample_split과 min_samples_leaf의 차이
    min_samples_split: "분할할 만큼 충분한 데이터가 있나?"
    min_samples_leaf: "분할해도 자식들이 너무 작지 않나?"
'''

In [7]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# 데이터 로드 및 전처리
df = pd.read_csv('datasets/heart.csv')
X = df.drop('output', axis=1)  # 독립 변수
y = df['output']              # 종속 변수 (심장병 여부: 1=있음, 0=없음)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 의사결정 나무 모델
dt = DecisionTreeClassifier(
    max_depth=20,          # 트리의 최대 깊이
    min_samples_split=10, # 노드를 분할하기 위한 최소 샘플 수
    min_samples_leaf=5,   # 리프 노드에 있어야 하는 최소 샘플 수
    random_state=42       # 결과 재현성을 위한 설정
)
dt.fit(X_train, y_train)  # 모델 학습

# 모델 평가
dt_preds = dt.predict(X_test)  # 테스트 데이터 예측
accuracy = accuracy_score(y_test, dt_preds)  # 정확도 계산
print("Decision Tree Classifier Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(y_test, dt_preds))


Decision Tree Classifier Accuracy: 0.7868852459016393

Classification Report:
               precision    recall  f1-score   support

           0       0.70      0.97      0.81        29
           1       0.95      0.62      0.75        32

    accuracy                           0.79        61
   macro avg       0.83      0.80      0.78        61
weighted avg       0.83      0.79      0.78        61



##### 로지스틱 회귀

In [ ]:
'''
데이터를 시그모이드 함수로 통해 0과 1사이의 확률로 분류하는 선형 모델 (분류도 됨)


주요 하이퍼파라미터

    penalty
        정규화 방식
        과적합 방지를 위해 가중치 크기를 제한
        'l1', 'l2', 'elasticnet', 'none'
        정규화 방식에 따라 solver의 사용이 제한됨
        
    C
        규제의 강도 (정규화의 반대값)
        0.1 , 1, 10 등등..
        너무 크면 과적합, 너무 작으면 과소적합 위험
        
    solver
        최적화 알고리즘 선택 (계산 방식)    
        'liblinear', 'saga', 'lbfgs'

    max_iter
        반복 학습 최대 횟수 (수렴 여부 결정)

정규화 방식과 solver

가장 기본이 되는 정규화 방식은 l2
l1정규화 방식은 희소한 모델을 만들 때 사용된다 - lbfgs 사용 불가
elasticnet 방식은 l1과 l2 정규화를 혼합한 방식 - saga만 사용 가능
'none'은 정규화 없이 학습하는 방식이며, liblinear 사용 불가함

'''

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 데이터 로드 및 전처리
df = pd.read_csv('datasets/heart.csv')
X = df.drop('output', axis=1)
y = df['output']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 로지스틱 회귀 모델
log_reg = LogisticRegression(
    C=1.0,           # 정규화 강도
    penalty='l2',    # L2 정규화
    solver='liblinear'  # 소규모 데이터셋에 적합한 솔버
)
log_reg.fit(X_train, y_train)  # 모델 학습

# 모델 평가
log_preds = log_reg.predict(X_test)  # 테스트 데이터 예측
accuracy = accuracy_score(y_test, log_preds)  # 정확도 계산
print("Logistic Regression Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(y_test, log_preds))

Logistic Regression Accuracy: 0.8688524590163934

Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.86      0.86        29
           1       0.88      0.88      0.88        32

    accuracy                           0.87        61
   macro avg       0.87      0.87      0.87        61
weighted avg       0.87      0.87      0.87        61



##### 랜덤 포레스트 - 횡적인 앙상블

In [ ]:
'''
여러 결정 트리를 앙상브라여 예측하는 모델.
주요 하이퍼파라미터
n_estimators
    결정 트리의 개수 - 투표하는 인원이 늘어날수록 안정적이지만 시간과 컴퓨팅 자원을 많이 소모함
    
max_depth
    각 트리가 얼마나 깊게 학습할지 설정
    깊을수록 복잡한 학습이 가능하지만 과적합의 우려가 음

min_samples_split
    노드를 분할하기 위해 필요한 최소 샘플 수 (분할 시도 전에 파악하는 sample 수)
    이 수 이하의 샘플을 가지면 분화하지 않음

min_samples_leaf
    리프노드에 있어야 하는 최소 샘플 수 (분할 수에 자식 노드의 sample 수 파악)
    자식 노드가 이 수 이하로 있으면 분할을 취소함 
'''

In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 데이터 로드, 전처리
df = pd.read_csv('datasets/heart.csv')
x = df.drop('output', axis = 1)
y = df['output']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state= 42)

# 랜덤 포레스트 분류 모델
rf = RandomForestClassifier(
    n_estimators=100,      # 트리 개수
    max_depth=10,          # 트리의 최대 깊이
    min_samples_split=5,   # 노드를 분할하기 위한 최소 샘플 수
    min_samples_leaf=3,    # 리프 노드에 있어야 하는 최소 샘플 수
    random_state=42        # 결과 재현성을 위한 설정
)
rf.fit(x_train, y_train)

# 모델 평가
rf_pred = rf.predict(x_test)
accuracy= accuracy_score(y_test, rf_pred)
print(accuracy, classification_report(y_test, rf_pred), sep='\n')

0.8360655737704918
              precision    recall  f1-score   support

           0       0.83      0.83      0.83        29
           1       0.84      0.84      0.84        32

    accuracy                           0.84        61
   macro avg       0.84      0.84      0.84        61
weighted avg       0.84      0.84      0.84        61



##### 그래디언트 부스트 - 종적인 앙상블

In [ ]:
'''
대부분의 파라미터는 랜덤 포레스트와 동일하며 하나의 차이가 있는데 바로

learning_rate 이다.
각 트리가 학습하는 속도를 조절하는 하이퍼 파라미터로,
작을수록 학습이 느리지만 일반화 성능이 좋을 수 있음
사실 학습률이 큰 경우 지역최소로 수렴하지 못하고 튕겨나갈 수가 있는것임


그래디언트 부스트 분류와 회귀 차이
    회귀는 'squared_error'의 손실함수 최소화
    분류는 'log_loss'라는 잘못 분류될 확률을 최소화한다.
'''

In [24]:
from sklearn.ensemble import GradientBoostingClassifier

# 데이터 로드 및 전처리
df = pd.read_csv('datasets/heart.csv')
X = df.drop('output', axis=1)
y = df['output']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 그래디언트 부스트 분류 모델
gb = GradientBoostingClassifier(
    learning_rate=0.001,     # 학습률
    n_estimators=500,      # 부스팅 반복 횟수 (트리 개수)
    max_depth=10,           # 각 트리의 최대 깊이
    min_samples_split=10,  # 노드를 분할하기 위한 최소 샘플 수
    min_samples_leaf=5,    # 리프 노드에 있어야 하는 최소 샘플 수
    random_state=42        # 결과 재현성을 위한 설정
)
gb.fit(X_train, y_train)  # 모델 학습

# 모델 평가
gb_preds = gb.predict(X_test)  # 테스트 데이터 예측
accuracy = accuracy_score(y_test, gb_preds)  # 정확도 계산
print("Gradient Boosting Classifier Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(y_test, gb_preds))

Gradient Boosting Classifier Accuracy: 0.819672131147541

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.79      0.81        29
           1       0.82      0.84      0.83        32

    accuracy                           0.82        61
   macro avg       0.82      0.82      0.82        61
weighted avg       0.82      0.82      0.82        61



##### 서포트 벡터 머신

In [ ]:
'''
서포트 벡터 머신은 데이터를 고차원 공간으로 변환해 분류하는 모델

주요 파라미터
C
    마진 오류에 대한 허용 정도(정규화 강도)
    값이 작을수록 마진을 넓게 잡고 과적합을 방지함
    C가 작으면 일부 틀려도 전체 균형을 봄. 
    C가 크면 틀림 없이 정확하게 맞추려 함(과적합)

kernel 
    사용할 커널 함수
    'linear', 'rbf', 'poly', sigmoid'
    
gamma
    커널 함수의 영향 범위
    값이 클수록 경계가 날카로움
    'scale', 'auto', 0.01, 1
    
degree
다향 커널('poly')에서 다항식의 차수

probability
    분류 결과를 확률로 출력할지 여부
    True, False
'''

In [34]:
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# 데이터 로드 및 전처리
df = pd.read_csv('datasets/heart.csv')  # heart.csv 파일 로드
X = df.drop('output', axis=1)  # 독립 변수 (특징 데이터)
y = df['output']               # 종속 변수 (1: 심장병 있음, 0: 없음)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# SVM 분류 모델 설정
svm = SVC(
    kernel='rbf',    # RBF 커널 (비선형 데이터 처리)
    C=3,             # 정규화 강도
    gamma='scale'    # 감마 설정 (특성 스케일에 따라 자동 설정)
)
svm.fit(X_train, y_train)  # 모델 학습

# 모델 평가
svm_preds = svm.predict(X_test)  # 테스트 데이터 예측
accuracy = accuracy_score(y_test, svm_preds)  # 정확도 계산
print("SVM Classifier Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(y_test, svm_preds))


SVM Classifier Accuracy: 0.7377049180327869

Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.62      0.69        29
           1       0.71      0.84      0.77        32

    accuracy                           0.74        61
   macro avg       0.75      0.73      0.73        61
weighted avg       0.74      0.74      0.73        61

